# 🥇 Gold Layer – Aggregations & Business Tables
**Chicago Crime Dataset – ETL Workflow Automation**

Reads clean Silver data and produces three Gold analytical tables:
- gold_crime_by_type_year  – crime counts by primary type & year
- gold_crime_by_location   – crime counts by location description
- gold_arrest_summary      – arrest rate metrics by type, year, and district

In [0]:
# ─────────────────────────────────────────────
# 1. PARAMETERS
# ─────────────────────────────────────────────
dbutils.widgets.text("silver_container",  "silver",                   "Silver Container")
dbutils.widgets.text("silver_path",       "chicago_crime/",           "Silver Path")
dbutils.widgets.text("gold_container",    "gold",                     "Gold Container")
dbutils.widgets.text("gold_base_path",    "chicago_crime/",           "Gold Base Path")
dbutils.widgets.text("storage_account",   "etlworkflowautomation",   "ADLS Storage Account")
dbutils.widgets.text("run_date",          "",                         "Run Date (YYYY-MM-DD)")
dbutils.widgets.text("pipeline_run_id",   "manual",                   "ADF Pipeline Run ID")

SILVER_CONTAINER  = dbutils.widgets.get("silver_container")
SILVER_PATH       = dbutils.widgets.get("silver_path")
GOLD_CONTAINER    = dbutils.widgets.get("gold_container")
GOLD_BASE_PATH    = dbutils.widgets.get("gold_base_path")
STORAGE_ACCOUNT   = dbutils.widgets.get("storage_account")
RUN_DATE          = dbutils.widgets.get("run_date")
PIPELINE_RUN_ID   = dbutils.widgets.get("pipeline_run_id")

In [0]:
# ─────────────────────────────────────────────
# 2. IMPORTS & SPARK SETUP
# ─────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, round as spark_round,
    when, lit, current_timestamp, max as spark_max, min as spark_min
)
from delta.tables import DeltaTable
from datetime import datetime, date
import traceback

spark = SparkSession.builder.appName("Gold_Chicago_Crime_Aggregation").getOrCreate()

SECRET_SCOPE = "etlworkflowautomation" 
SECRET_KEY   = "***************************************************"  
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SECRET_KEY
)

RUN_DATE   = RUN_DATE if RUN_DATE else str(date.today())
START_TIME = datetime.now()
print(f"Run date : {RUN_DATE}")

Run date : 2026-05-20

In [0]:
# ─────────────────────────────────────────────
# 3. READ SILVER
# ─────────────────────────────────────────────
silver_uri = f"wasbs://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/{SILVER_PATH}"

try:
    df_silver = spark.read.format("delta").load(silver_uri)
    silver_count = df_silver.count()
    print(f"Silver rows loaded: {silver_count:,}")
except Exception as e:
    print(f"Silver read failed: {e}")
    dbutils.notebook.exit(f"FAILED|0|{str(e)}")

# Cache since we derive multiple aggregations
df_silver.cache()
print("Silver DataFrame cached.")

Silver rows loaded: 8,554,479
Silver DataFrame cached.

In [0]:
# ─────────────────────────────────────────────
# HELPER – write Gold Delta table
# ─────────────────────────────────────────────
def write_gold_table(df, table_name: str, merge_keys: list):
    """Upsert aggregated DataFrame into a Gold Delta table."""
    path = f"wasbs://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/{GOLD_BASE_PATH}{table_name}/"
    df_out = (
        df
        .withColumn("_updated_at",       current_timestamp())
        .withColumn("_pipeline_run_id",  lit(PIPELINE_RUN_ID))
    )
    try:
        if DeltaTable.isDeltaTable(spark, path):
            dt = DeltaTable.forPath(spark, path)
            cond = " AND ".join([f"existing.{k} = incoming.{k}" for k in merge_keys])
            (
                dt.alias("existing")
                .merge(df_out.alias("incoming"), cond)
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
            print(f"UPSERT → {table_name}")
        else:
            df_out.write.format("delta").mode("overwrite").save(path)
            print(f"CREATE → {table_name}")

        return df_out.count()
    except Exception as e:
        print(f"FAILED → {table_name}: {e}")
        traceback.print_exc()
        raise

In [0]:
# ─────────────────────────────────────────────
# 4a. GOLD TABLE 1 – Crime Count by Type & Year
# ─────────────────────────────────────────────
df_type_year = (
    df_silver
    .groupBy("primary_type", "crime_year")
    .agg(
        count("*").alias("total_crimes"),
        spark_sum(when(col("arrest") == True, 1).otherwise(0)).alias("arrests"),
        spark_sum(when(col("domestic") == True, 1).otherwise(0)).alias("domestic_crimes"),
        spark_sum(when(col("is_night") == True, 1).otherwise(0)).alias("night_crimes"),
        spark_sum(when(col("is_weekend") == True, 1).otherwise(0)).alias("weekend_crimes"),
    )
    .withColumn("arrest_rate",
        spark_round(col("arrests") / col("total_crimes") * 100, 2)
    )
)

n1 = write_gold_table(df_type_year, "gold_crime_by_type_year", ["primary_type", "crime_year"])
print(f"  Rows: {n1:,}")
df_type_year.show(10, truncate=False)

UPSERT → gold_crime_by_type_year
 Rows: 799
+-------------------+----------+------------+-------+---------------+------------+--------------+-----------+
primary_type |crime_year|total_crimes|arrests|domestic_crimes|night_crimes|weekend_crimes|arrest_rate|
+-------------------+----------+------------+-------+---------------+------------+--------------+-----------+
MOTOR VEHICLE THEFT|2006 |21818 |1988 |53 |9916 |6031 |9.11 |
OTHER OFFENSE |2003 |31149 |5856 |10925 |10709 |8355 |18.8 |
BURGLARY |2003 |25157 |1615 |230 |8467 |5952 |6.42 |
ARSON |2003 |955 |140 |53 |601 |287 |14.66 |
OBSCENITY |2004 |13 |12 |0 |4 |4 |92.31 |
PUBLIC INDECENCY |2005 |4 |4 |0 |2 |1 |100.0 |
PUBLIC INDECENCY |2001 |9 |9 |0 |5 |3 |100.0 |
BATTERY |2006 |80666 |18892 |40907 |35288 |25674 |23.42 |
ROBBERY |2006 |15969 |1734 |232 |7606 |4562 |10.86 |
INTIMIDATION |2007 |255 |51 |43 |64 |61 |20.0 |
+-------------------+----------+------------+-------+---------------+------------+--------------+-----------+
only showing top 10 rows

In [0]:
# ─────────────────────────────────────────────
# 4b. GOLD TABLE 2 – Crime Count by Location
# ─────────────────────────────────────────────
df_location = (
    df_silver
    .groupBy("location_description")
    .agg(
        count("*").alias("total_crimes"),
        spark_sum(when(col("arrest") == True, 1).otherwise(0)).alias("arrests"),
        spark_round(
            spark_sum(when(col("arrest") == True, 1).otherwise(0)) / count("*") * 100, 2
        ).alias("arrest_rate"),
        spark_sum(when(col("domestic") == True, 1).otherwise(0)).alias("domestic_crimes"),
    )
    .orderBy(col("total_crimes").desc())
)

n2 = write_gold_table(df_location, "gold_crime_by_location", ["location_description"])
print(f"  Rows: {n2:,}")
df_location.show(10, truncate=False)

UPSERT → gold_crime_by_location
 Rows: 219
+------------------------------+------------+-------+-----------+---------------+
location_description |total_crimes|arrests|arrest_rate|domestic_crimes|
+------------------------------+------------+-------+-----------+---------------+
STREET |2235909 |586744 |26.24 |182506 |
RESIDENCE |1398885 |183212 |13.1 |536149 |
APARTMENT |1027789 |155071 |15.09 |483558 |
SIDEWALK |768112 |373731 |48.66 |84193 |
OTHER |269918 |49378 |18.29 |18733 |
PARKING LOT/GARAGE(NON.RESID.)|202914 |39953 |19.69 |10276 |
ALLEY |190044 |79601 |41.89 |16820 |
SMALL RETAIL STORE |174001 |42480 |24.41 |2312 |
SCHOOL, PUBLIC, BUILDING |146365 |44300 |30.27 |3274 |
RESTAURANT |144562 |26578 |18.39 |4627 |
+------------------------------+------------+-------+-----------+---------------+
only showing top 10 rows

In [0]:
# ─────────────────────────────────────────────
# 4c. GOLD TABLE 3 – Arrest Summary by Type & Hour
# ─────────────────────────────────────────────
df_arrest = (
    df_silver
    .groupBy("primary_type", "crime_hour")
    .agg(
        count("*").alias("total_crimes"),
        spark_sum(when(col("arrest") == True, 1).otherwise(0)).alias("total_arrests"),
        spark_round(
            spark_sum(when(col("arrest") == True, 1).otherwise(0)) / count("*") * 100, 2
        ).alias("arrest_rate_pct"),
        spark_sum(when(col("is_night") == True, 1).otherwise(0)).alias("night_crimes"),
    )
)

n3 = write_gold_table(df_arrest, "gold_arrest_summary", ["primary_type", "crime_hour"])
print(f"  Rows: {n3:,}")
df_arrest.show(10, truncate=False)

UPSERT → gold_arrest_summary
 Rows: 770
+--------------------------------+----------+------------+-------------+---------------+------------+
primary_type |crime_hour|total_crimes|total_arrests|arrest_rate_pct|night_crimes|
+--------------------------------+----------+------------+-------------+---------------+------------+
WEAPONS VIOLATION |23 |9603 |7372 |76.77 |9603 |
OBSCENITY |14 |56 |41 |73.21 |0 |
OBSCENITY |11 |41 |28 |68.29 |0 |
SEX OFFENSE |18 |1602 |452 |28.21 |0 |
ROBBERY |14 |13327 |1648 |12.37 |0 |
HOMICIDE |22 |804 |360 |44.78 |804 |
CRIMINAL TRESPASS |1 |6128 |4113 |67.12 |6128 |
OFFENSE INVOLVING CHILDREN |11 |2076 |405 |19.51 |0 |
GAMBLING |6 |33 |33 |100.0 |0 |
INTERFERENCE WITH PUBLIC OFFICER|20 |1561 |1449 |92.83 |1561 |
+--------------------------------+----------+------------+-------------+---------------+------------+
only showing top 10 rows

In [0]:
# ─────────────────────────────────────────────
# 5. UNPERSIST CACHE
# ─────────────────────────────────────────────
df_silver.unpersist()
print("Silver cache released.")

Silver cache released.

In [0]:
# ─────────────────────────────────────────────
# 6. SUMMARY & EXIT
# ─────────────────────────────────────────────
total_gold_rows = n1 + n2 + n3
duration_sec = (datetime.now() - START_TIME).total_seconds()

print("\n── Gold Layer Summary ───────────────────")
print(f"  gold_crime_by_type_year  : {n1:>8,} rows")
print(f"  gold_crime_by_location   : {n2:>8,} rows")
print(f"  gold_arrest_summary      : {n3:>8,} rows")
print(f"  Duration                 : {duration_sec:>8.1f} s")
print("─────────────────────────────────────────")

dbutils.notebook.exit(f"SUCCESS|{total_gold_rows}|{duration_sec:.1f}")